# Gap Fill — Targeted Generation for Weak Labels

Loads all classified chunks from all 5 documents and re-runs generation specifically targeting Adaptability, Personal Life, Timing, and Psychology. The original classification was correct for its purpose — but a chunk about Buffett's See's Candy acquisition (classified Strategy Development) also contains timing reasoning, personal philosophy, and psychological discipline that the original label-specific prompt didn't extract.

This notebook temporarily overrides each chunk's label to force generation from a different angle. The assembly notebook's dedup catches any overlap with existing pairs.

In [1]:
import sys, re, statistics, importlib, asyncio
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS,
    generate_pairs, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
    _parse_llm_json, PRIMARY_MODEL,
)
from litellm import acompletion

## Load all classified chunks from every document
These chunks already went through classification and generation once. We're re-using them with different label-specific prompts to extract content the first pass missed.

In [2]:
CHECKPOINTS = {
    "Florida Speech": "florida_classified",
    "Ivey 2008": "ivey_classified",
    "Partnership Letters": "letters_classified",
    "Cunningham Essays": "cunningham_classified",
    "Notre Dame": "notredame_classified",
}

all_chunks = []
for name, ckpt in CHECKPOINTS.items():
    try:
        chunks = load_checkpoint(ckpt)
        all_chunks.extend(chunks)
        print(f"  ✓ {name}: {len(chunks)} chunks")
    except FileNotFoundError:
        print(f"  ✗ {name}: not found")

print(f"\nTotal chunks available: {len(all_chunks)}")

  ✓ Florida Speech: 27 chunks
  ✓ Ivey 2008: 12 chunks
  ✓ Partnership Letters: 122 chunks
  ✓ Cunningham Essays: 131 chunks
  ✓ Notre Dame: 35 chunks

Total chunks available: 327


## Targeted generation strategy

For each gap label, we send every chunk through a specialized prompt that asks: "Is there anything in this passage relevant to [label]? If yes, generate pairs. If no, return an empty array."

This is different from the original generation — which assumed the chunk belonged to its classified label and generated accordingly. Here we're explicitly asking the LLM to look for secondary content it may have ignored the first time.

We use a screening prompt first to avoid wasting generation calls on chunks with zero relevance to the target label.

In [3]:
GAP_LABELS = ["Adaptability", "Personal Life", "Timing", "Psychology"]

TARGETED_PROMPT = """You are extracting question-answer pairs about Warren Buffett from a knowledge source that may or may not contain content relevant to a SPECIFIC label.

TARGET LABEL: {label}
LABEL FOCUS: {label_description}
SUBLABELS TO LOOK FOR: {sublabels}

KNOWLEDGE SOURCE (use for facts, never reference directly):
---
{chunk_text}
---

INSTRUCTIONS:
1. Read the source carefully for ANY content relevant to the target label, even if the source is primarily about something else.
2. If you find relevant content, generate 1-3 Q&A pairs grounded in it.
3. If the source has NOTHING relevant to this label, respond with an empty array: []
4. Questions must sound like real chatbot queries — short, natural, direct. Not academic essay prompts.
5. Answers must be self-contained. NEVER reference "the passage", "the text", "the source", or any underlying document. Write as if speaking from internalized knowledge.
6. Match answer depth to question complexity: 1-2 sentences for factual, 3-5 for reasoning, 5-7 for analytical.
7. Anchor answers in specific names, numbers, and examples from the source.

Respond with ONLY a JSON array (empty if nothing relevant):
[{{"question": "...", "answer": "...", "sublabel": "..."}}]"""

async def targeted_generate(chunk, target_label):
    """Generate pairs for a specific label from any chunk, regardless of its original classification."""
    lcfg = LABELS[target_label]
    try:
        response = await acompletion(
            model=PRIMARY_MODEL,
            messages=[{"role": "user", "content": TARGETED_PROMPT.format(
                label=target_label,
                label_description=lcfg["description"],
                sublabels=", ".join(lcfg["sublabels"]),
                chunk_text=chunk.text[:3000],
            )}],
            temperature=0.7,
            max_tokens=2000,
        )
        pairs_data = _parse_llm_json(response.choices[0].message.content)
        if not pairs_data:
            return []
        return [
            QAPair(
                question=p["question"], answer=p["answer"], label=target_label,
                sublabel=p.get("sublabel"), source_chunk_id=chunk.chunk_id,
                source_file=chunk.source_file, generation_model=PRIMARY_MODEL,
            )
            for p in pairs_data if "question" in p and "answer" in p
        ]
    except Exception as e:
        return []

## Run targeted generation for all gap labels
Iterates through each gap label and sends all chunks through the targeted prompt. Most chunks will return empty arrays — that's expected and cheap. The ones that do return pairs are new content the original pass missed.

In [4]:
all_gap_pairs = []

for label in GAP_LABELS:
    print(f"\n{'='*60}")
    print(f"Targeting: {label}")
    print(f"{'='*60}")

    label_pairs = []
    batch_size = 10

    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i + batch_size]
        results = await asyncio.gather(*[targeted_generate(c, label) for c in batch])
        for pairs in results:
            label_pairs.extend(pairs)
        done = min(i + batch_size, len(all_chunks))
        hits = len(label_pairs)
        print(f"  {done}/{len(all_chunks)} chunks scanned | {hits} pairs found so far")

    all_gap_pairs.extend(label_pairs)
    print(f"  → {label}: {len(label_pairs)} new pairs")

save_checkpoint(all_gap_pairs, "gap_pairs_raw")
print(f"\nTotal gap pairs generated: {len(all_gap_pairs)}")


Targeting: Adaptability
  10/327 chunks scanned | 2 pairs found so far
  20/327 chunks scanned | 9 pairs found so far
  30/327 chunks scanned | 16 pairs found so far
  40/327 chunks scanned | 19 pairs found so far
  50/327 chunks scanned | 32 pairs found so far
  60/327 chunks scanned | 44 pairs found so far
  70/327 chunks scanned | 56 pairs found so far
  80/327 chunks scanned | 65 pairs found so far
  90/327 chunks scanned | 67 pairs found so far
  100/327 chunks scanned | 67 pairs found so far
  110/327 chunks scanned | 71 pairs found so far
  120/327 chunks scanned | 76 pairs found so far
  130/327 chunks scanned | 81 pairs found so far
  140/327 chunks scanned | 83 pairs found so far
  150/327 chunks scanned | 85 pairs found so far
  160/327 chunks scanned | 87 pairs found so far
  170/327 chunks scanned | 90 pairs found so far
  180/327 chunks scanned | 92 pairs found so far
  190/327 chunks scanned | 96 pairs found so far
  200/327 chunks scanned | 101 pairs found so far
  210

## Score and filter
Same quality gate as the document notebooks. Pairs must score above 0.7 composite to survive.

In [10]:
chunk_map = {c.chunk_id: c for c in all_chunks}
scored = await score_all(all_gap_pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "gap_pairs_filtered")

  Scored 10/756
  Scored 20/756
  Scored 30/756
  Scored 40/756
  Scored 50/756
  Scored 60/756
  Scored 70/756
  Scored 80/756
  Scored 90/756
  Scored 100/756
  Scored 110/756
  Scored 120/756
  Scored 130/756
  Scored 140/756
  Scored 150/756
  Scored 160/756
  Scored 170/756
  Scored 180/756
  Scored 190/756
  Scored 200/756
  Scored 210/756
  Scored 220/756
  Scored 230/756
  Scored 240/756
  Scored 250/756
  Scored 260/756
  Scored 270/756
  Scored 280/756
  Scored 290/756
  Scored 300/756
  Scored 310/756
  Scored 320/756
  Scored 330/756
  Scored 340/756
  Scored 350/756
  Scored 360/756
  Scored 370/756
  Scored 380/756
  Scored 390/756
  Scored 400/756
  Scored 410/756
  Scored 420/756
  Scored 430/756
  Scored 440/756
  Scored 450/756
  Scored 460/756
  Scored 470/756
  Scored 480/756
  Scored 490/756
  Scored 500/756
  Scored 510/756
  Scored 520/756
  Scored 530/756
  Scored 540/756
  Scored 550/756
  Scored 560/756
  Scored 570/756
  Scored 580/756
  Scored 590/756
  Scor

In [11]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Gap pairs after filtering: {len(filtered)} / {len(all_gap_pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(all_gap_pairs))*100:.1f}%\n")
if scores:
    print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
    print(f"Mean:   {statistics.mean(scores):.2f}")
    print(f"Median: {statistics.median(scores):.2f}")

Gap pairs after filtering: 579 / 756 raw
Drop rate: 23.4%

Score range: 0.70 — 0.93
Mean:   0.82
Median: 0.83


## Gap fill contribution
How many new pairs did we extract for each gap label? These get picked up automatically by the assembly notebook's checkpoint loader.

In [12]:
print("Gap fill contribution:\n")
gap_dist = Counter(p.label for p in filtered)
for label in GAP_LABELS:
    count = gap_dist.get(label, 0)
    print(f"  {label:25s} {count:3d} new pairs")
print(f"\n  Total: {len(filtered)} new pairs")

Gap fill contribution:

  Adaptability              114 new pairs
  Personal Life              58 new pairs
  Timing                    183 new pairs
  Psychology                224 new pairs

  Total: 579 new pairs


In [13]:
export_csv(filtered, Path("../intermediate/gap_qa.csv"))
export_detailed(filtered, Path("../intermediate/gap_qa_detailed.csv"))

Exported 579 pairs to ..\intermediate\gap_qa.csv
  Psychology: 224
  Timing: 183
  Adaptability: 114
  Personal Life: 58
Detailed export: ..\intermediate\gap_qa_detailed.csv


In [9]:
# all_gap_pairs = load_checkpoint("gap_pairs_raw")
# filtered = load_checkpoint("gap_pairs_filtered")